# Ejercicio Módulo 2
**Inteligencia Artificial - CEIA - FIUBA**

**César Alejandro Arana Castro**

En este ejercicio deben implementar un algoritmo de búsqueda que no sea **Búsqueda Primero en Anchura (BFS)** para resolver el problema de la Torre de Hanoi. La nota máxima dependerá del algoritmo implementado:

- **Búsqueda Primero en Profundidad**: nota máxima 6.
- **Búsqueda de Costo Uniforme**: nota máxima 6.
- **Búsqueda de Profundidad Limitada con Profundidad Iterativa**: nota máxima 7.
- **Búsqueda Voraz usando la heurística dada en el aula virtual**: nota máxima 8.
- **Búsqueda Voraz usando una heurística desarrollada por vos**: nota máxima 9.
- **Búsqueda A\* usando la heurística dada en el aula virtual**: nota máxima 9.
- **Búsqueda A\* usando una heurística desarrollada por vos**: nota máxima 10.

La función debe devolver la salida correspondiente a la solución encontrada o `None si no se encontró una solución.

Además, debe calcular métricas de rendimiento que, como mínimo, incluyan:

- `solution_found`: `True` si se encontró la solución, `False` en caso contrario.
- `nodes_explored`: cantidad de nodos explorados (entero).
- `states_visited`: cantidad de estados distintos visitados (entero).
- `nodes_in_frontier`: cantidad de nodos que quedaron en la frontera al finalizar la ejecución (entero).
- `max_depth`: máxima profundidad explorada (entero).
- `cost_total`: costo total para encontrar la solución (float).

In [1]:
from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi
from aima_libs.tree_hanoi import NodeHanoi

In [ ]:
from aima_libs.aima import PriorityQueue
def search_algorithm(number_disks=5) -> tuple[NodeHanoi, dict]:

    list_disks = [i for i in range(number_disks, 0, -1)]
    initial_state = StatesHanoi(list_disks, [], [], max_disks=number_disks)
    goal_state = StatesHanoi([], [], list_disks, max_disks=number_disks)
    problem = ProblemHanoi(initial=initial_state, goal=goal_state)

    # Heurística
    # h(s) = 0 si s es el estado objetivo; en otro caso, h(s) = B(s) + 1, donde B(s) cuenta los discos encima del disco pendiente seleccionado.
    def heuristic(state):
        # Estima movimientos necesarios para avanzar a la pila objetivo en la varilla C.
        if problem.goal_test(state):
            return 0

        target_rod = state.rods[2]

        # El primer disco de la meta(agrupamiento en la varilla C) que no ocupa su posición desde la base es el disco pendiente seleccionado por esta heurística.
        selected_disk = None
        for index, expected_disk in enumerate(list_disks):
            if index >= len(target_rod) or target_rod[index] != expected_disk:
                selected_disk = expected_disk
                break

        # Se cuentan los discos que están encima del disco seleccionado en su varilla.
        for rod in state.rods:
            if selected_disk in rod:
                disk_index = rod.index(selected_disk)
                disks_above = len(rod) - disk_index - 1
                return disks_above + 1

        raise ValueError("No se encontró el disco pendiente en el estado.")

    initial_node = NodeHanoi(initial_state)
    frontier = PriorityQueue(
        order="min",
        f=lambda node: node.path_cost + heuristic(node.state),
    )
    frontier.append(initial_node)

    # Guarda el menor costo g conocido para cada estado, evitando ciclos
    # y permitiendo reabrir un estado si se encuentra un camino más barato.
    best_cost = {initial_state: 0.0}
    solution = None
    metrics = {
        "solution_found": False,
        "nodes_explored": 0,
        "states_visited": 1,
        "nodes_in_frontier": 1,
        "max_depth": 0,
        "cost_total": 0.0,
    }

    while frontier:
        _, node = frontier.pop()

        # Ignoramos entradas antiguas si ya se encontró un camino más barato.
        if node.path_cost > best_cost.get(node.state, float("inf")):
            continue

        metrics["nodes_explored"] += 1
        metrics["max_depth"] = max(metrics["max_depth"], node.depth)

        if problem.goal_test(node.state):
            solution = node
            metrics["solution_found"] = True
            metrics["cost_total"] = float(node.path_cost)
            break

        for child in node.expand(problem):
            previous_cost = best_cost.get(child.state)
            if previous_cost is None:
                metrics["states_visited"] += 1

            if previous_cost is None or child.path_cost < previous_cost:
                best_cost[child.state] = child.path_cost
                frontier.append(child)

    metrics["nodes_in_frontier"] = len(frontier)
    assert solution is not None, "No se encontró una solución para la Torre de Hanoi."

    return solution, metrics

Se prueba la función:

In [3]:
solution, metrics = search_algorithm(number_disks=5)

Veamos las métricas:

In [4]:
for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 194
states_visited: 212
nodes_in_frontier: 18
max_depth: 31
cost_total: 31.0


Veamos el camino de estados desde el principio a la solución:

In [5]:
for nodos in solution.path():
    print(nodos)

<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 2 |  | 1>
<Node HanoiState: 5 4 3 | 2 | 1>
<Node HanoiState: 5 4 3 | 2 1 | >
<Node HanoiState: 5 4 | 2 1 | 3>
<Node HanoiState: 5 4 1 | 2 | 3>
<Node HanoiState: 5 4 1 |  | 3 2>
<Node HanoiState: 5 4 |  | 3 2 1>
<Node HanoiState: 5 | 4 | 3 2 1>
<Node HanoiState: 5 | 4 1 | 3 2>
<Node HanoiState: 5 2 | 4 1 | 3>
<Node HanoiState: 5 2 1 | 4 | 3>
<Node HanoiState: 5 2 1 | 4 3 | >
<Node HanoiState: 5 2 | 4 3 | 1>
<Node HanoiState: 5 | 4 3 2 | 1>
<Node HanoiState: 5 | 4 3 2 1 | >
<Node HanoiState:  | 4 3 2 1 | 5>
<Node HanoiState: 1 | 4 3 2 | 5>
<Node HanoiState: 1 | 4 3 | 5 2>
<Node HanoiState:  | 4 3 | 5 2 1>
<Node HanoiState: 3 | 4 | 5 2 1>
<Node HanoiState: 3 | 4 1 | 5 2>
<Node HanoiState: 3 2 | 4 1 | 5>
<Node HanoiState: 3 2 1 | 4 | 5>
<Node HanoiState: 3 2 1 |  | 5 4>
<Node HanoiState: 3 2 |  | 5 4 1>
<Node HanoiState: 3 | 2 | 5 4 1>
<Node HanoiState: 3 | 2 1 | 5 4>
<Node HanoiState:  | 2 1 | 5 4 3>
<Node HanoiState: 1 | 2 | 5 4 

Y las acciones que el agente debería aplicar para llegar al objetivo:

In [6]:
for act in solution.solution():
    print(act)

Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 4 from 1 to 2
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 3 from 3 to 2
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 5 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 3 from 2 to 1
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 4 from 2 to 3
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
